In [2]:
import pandas as pd

df = pd.read_csv("../data/interim/aneel_eletropaulo.csv", encoding="utf-8", na_values=["", " ", "NA", "N/A", "null"])

df.head()

,DatGeracaoConjuntoDados,IdeConjuntoUnidadeConsumidora,DscConjuntoUnidadeConsumidora,DscAlimentadorSubestacao,DscSubestacaoDistribuicao,NumOrdemInterrupcao,DscTipoInterrupcao,IdeMotivoInterrupcao,DatInicioInterrupcao,DatFimInterrupcao,DscFatoGeradorInterrupcao,NumNivelTensao,NumUnidadeConsumidora,NumConsumidorConjunto,NumAno,NomAgenteRegulado,SigAgente,NumCPFCNPJ
0,2026-04-30,12952,PARELHEIROS,PRE 0111,PRE,5308841-1,Não Programada,0,2018-01-11 18:41:10,2018-01-11 22:15:25,Interna - Nao Programada - Proprias do sistema...,13800,40,35138,2018,ELETROPAULO METROPOLITANA ELETRICIDADE DE SAO ...,ELETROPAULO,61695227000193
1,2026-04-30,13000,VITÓRIA,VIT 0107,VIT,5339907-1,Não Programada,0,2018-01-21 22:57:31,2018-01-22 07:47:38,Interna - Nao Programada - Proprias do sistema...,240,1,97131,2018,ELETROPAULO METROPOLITANA ELETRICIDADE DE SAO ...,ELETROPAULO,61695227000193
2,2026-04-30,12876,CARAPICUIBA,CPI 0112,CPI,5371620-1,Não Programada,0,2018-01-31 15:48:27,2018-01-31 17:29:48,Interna - Nao Programada - Proprias do sistema...,240,1,121701,2018,ELETROPAULO METROPOLITANA ELETRICIDADE DE SAO ...,ELETROPAULO,61695227000193
3,2026-04-30,12928,JUQUITIBA,JUQ 0102,JUQ,5402739-1,Não Programada,0,2018-02-14 00:36:33,2018-02-14 08:18:23,Interna - Nao Programada - Proprias do sistema...,13800,1,17997,2018,ELETROPAULO METROPOLITANA ELETRICIDADE DE SAO ...,ELETROPAULO,61695227000193
4,2026-04-30,12895,CONGONHAS,COG 0106,COG,5438990-1,Não Programada,0,2018-02-27 13:20:44,2018-02-27 15:59:22,Interna - Nao Programada - Proprias do sistema...,240,1,55060,2018,ELETROPAULO METROPOLITANA ELETRICIDADE DE SAO ...,ELETROPAULO,61695227000193


In [ ]:
df['DatInicioInterrupcao'] = pd.to_datetime(df['DatInicioInterrupcao'])
df['DatFimInterrupcao']    = pd.to_datetime(df['DatFimInterrupcao'])
df['duracao_horas']        = (df['DatFimInterrupcao'] - df['DatInicioInterrupcao']).dt.total_seconds() / 3600

# Verificar siglas disponíveis antes de filtrar
print(df['SigAgente'].value_counts())

SigAgente
ELETROPAULO    2570618
Name: count, dtype: int64


In [4]:
df_sp_nao_prog = df[df['DscTipoInterrupcao'] == 'Não Programada'].copy()


In [24]:
df_sp_nao_prog.loc[df_sp_nao_prog['duracao_horas'].idxmax(), ['DatInicioInterrupcao','DatFimInterrupcao','IdeMotivoInterrupcao','NumUnidadeConsumidora','duracao_horas','DscFatoGeradorInterrupcao']]

DatInicioInterrupcao                                       2020-06-02 16:03:00
DatFimInterrupcao                                          2020-07-10 10:40:29
IdeMotivoInterrupcao                                                         0
NumUnidadeConsumidora                                                        2
duracao_horas                                                       906.624722
DscFatoGeradorInterrupcao    Interna - Nao Programada - Proprias do sistema...
Name: 575354, dtype: object

In [6]:
# Separar a coluna em 4 níveis, aceitando ';' e ' - ' como delimitadores
fatores = (
    df_sp_nao_prog['DscFatoGeradorInterrupcao']
    .fillna('')
    .str.strip()
    .str.lower()
    .str.split(r'\s*(?:;|\s-\s)\s*', n=3, expand=True)
)

fatores.columns = [
    'fator_1',
    'fator_2',
    'fator_3',
    'fator_4',
]

for coluna in fatores.columns:
    df_sp_nao_prog[coluna] = fatores[coluna].replace('', pd.NA)

df_sp_nao_prog[['DscFatoGeradorInterrupcao', 'fator_1', 'fator_2', 'fator_3', 'fator_4']].head()


,DscFatoGeradorInterrupcao,fator_1,fator_2,fator_3,fator_4
0,Interna - Nao Programada - Proprias do sistema...,interna,nao programada,proprias do sistema,falha de material ou equipamento
1,Interna - Nao Programada - Proprias do sistema...,interna,nao programada,proprias do sistema,sobrecarga
2,Interna - Nao Programada - Proprias do sistema...,interna,nao programada,proprias do sistema,falha de material ou equipamento
3,Interna - Nao Programada - Proprias do sistema...,interna,nao programada,proprias do sistema,falha de material ou equipamento
4,Interna - Nao Programada - Proprias do sistema...,interna,nao programada,proprias do sistema,falha de material ou equipamento


In [7]:
print(df['DscTipoInterrupcao'].value_counts(normalize=True))
print(df['IdeMotivoInterrupcao'].value_counts())

DscTipoInterrupcao
Não Programada    0.956876
Programada        0.043124
Name: proportion, dtype: float64
IdeMotivoInterrupcao
0    2295980
6     148666
3     122931
8       1933
7       1108
Name: count, dtype: int64


In [8]:
df_sp_nao_prog.loc[:, 'mes'] = df_sp_nao_prog['DatInicioInterrupcao'].dt.month
df_sp_nao_prog.loc[:, 'ano'] = df_sp_nao_prog['DatInicioInterrupcao'].dt.year
pivot = df_sp_nao_prog.pivot_table(index='mes', columns='ano', aggfunc='size', fill_value=0)

In [9]:
df_sp_nao_prog.loc[:, 'consumer_hours'] = (
    df_sp_nao_prog['duracao_horas'] * df_sp_nao_prog['NumUnidadeConsumidora']
)

In [10]:
df_regular     = df_sp_nao_prog[df_sp_nao_prog['IdeMotivoInterrupcao'] == 0]
df_dia_critico = df_sp_nao_prog[df_sp_nao_prog['IdeMotivoInterrupcao'] == 6]

# Total de UCs do conjunto (denominador do DEC)
total_ucs = df_sp_nao_prog['NumConsumidorConjunto'].max()  # ajustar se necessário

dec_oficial    = df_regular.groupby('ano').apply(
    lambda x: (x['duracao_horas'] * x['NumUnidadeConsumidora']).sum() / total_ucs
)
dec_expurgado  = df_dia_critico.groupby('ano').apply(
    lambda x: (x['duracao_horas'] * x['NumUnidadeConsumidora']).sum() / total_ucs
)
dec_real       = dec_oficial.add(dec_expurgado, fill_value=0)
gap_percentual = dec_expurgado / dec_real * 100

C:\Users\Andre\AppData\Local\Temp\ipykernel_1196\1919941013.py:7: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  dec_oficial    = df_regular.groupby('ano').apply(
C:\Users\Andre\AppData\Local\Temp\ipykernel_1196\1919941013.py:10: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  dec_expurgado  = df_dia_critico.groupby('ano').apply(


In [11]:
df_sp_nao_prog.columns

Index(['DatGeracaoConjuntoDados', 'IdeConjuntoUnidadeConsumidora',
       'DscConjuntoUnidadeConsumidora', 'DscAlimentadorSubestacao',
       'DscSubestacaoDistribuicao', 'NumOrdemInterrupcao',
       'DscTipoInterrupcao', 'IdeMotivoInterrupcao', 'DatInicioInterrupcao',
       'DatFimInterrupcao', 'DscFatoGeradorInterrupcao', 'NumNivelTensao',
       'NumUnidadeConsumidora', 'NumConsumidorConjunto', 'NumAno',
       'NomAgenteRegulado', 'SigAgente', 'NumCPFCNPJ', 'duracao_horas',
       'fator_1', 'fator_2', 'fator_3', 'fator_4', 'mes', 'ano',
       'consumer_hours'],
      dtype='object')

In [12]:
termos_climaticos = [
    'vento', 'chuva', 'tempestade','raio', 
    'granizo', 'temporal', 'arvore',
    'falha de material', 'sobrecarga'
]
padrao = '|'.join(termos_climaticos)
df_sp_nao_prog['causa_climatica'] = (
    df_sp_nao_prog['DscFatoGeradorInterrupcao']
    .str.lower()
    .str.contains(padrao, na=False)
)
print(df_sp_nao_prog['causa_climatica'].value_counts(normalize=True))

# Frequência por termo
for t in termos_climaticos:
    n = df_sp_nao_prog['DscFatoGeradorInterrupcao'].str.lower().str.contains(t, na=False).sum()
    print(f"{t}: {n}")

causa_climatica
True     0.767471
False    0.232529
Name: proportion, dtype: float64
vento: 203045
chuva: 0
tempestade: 0
raio: 0
granizo: 0
temporal: 0
arvore: 306161
falha de material: 1142419
sobrecarga: 236170


In [13]:
df_sp_nao_prog['fator_1'].value_counts(normalize=True)

fator_1
interna    0.998431
externa    0.001569
Name: proportion, dtype: float64

In [14]:
df_sp_nao_prog['fator_3'].value_counts(normalize=True)

fator_3
proprias do sistema    0.569968
meio ambiente          0.217149
terceiros              0.145503
falha operacional      0.056625
nao classificada       0.010194
alivio de carga        0.000450
alteracao              0.000110
Name: proportion, dtype: float64

In [15]:
df_sp_nao_prog['fator_4'].value_counts(normalize=True)

fator_4
falha de material ou equipamento                             0.469440
arvore ou vegetacao                                          0.125807
interferencia de terceiros                                   0.100929
sobrecarga                                                   0.097046
vento                                                        0.083435
erro de operacao                                             0.037214
roubo                                                        0.025401
servico mal executado                                        0.019976
objeto na rede                                               0.010877
nao identificada                                             0.007145
animais                                                      0.005157
descarga atmosferica                                         0.003879
abalroamento                                                 0.003566
ligacao clandestina                                          0.002940
desligamento

In [16]:
pareto = (
    df_sp_nao_prog
    .groupby('DscAlimentadorSubestacao')
    .agg(
        n_eventos=('duracao_horas', 'count'),
        duracao_total=('duracao_horas', 'sum'),
        consumer_hours=('consumer_hours', 'sum')
    )
    .sort_values('duracao_total', ascending=False)
)
pareto['duracao_acum_pct'] = pareto['duracao_total'].cumsum() / pareto['duracao_total'].sum()

In [18]:
comparativo = df_sp_nao_prog.groupby(
    ['SigAgente', 'ano']
).apply(
    lambda x: (x['duracao_horas'] * x['NumUnidadeConsumidora']).sum()
).reset_index(name='consumer_hours_total')

C:\Users\Andre\AppData\Local\Temp\ipykernel_1196\2020649355.py:3: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  ).apply(


In [21]:
apagao_out2024 = df[
    (df['DatInicioInterrupcao'] >= '2024-10-10') &
    (df['DatInicioInterrupcao'] <= '2024-10-16')
]
print(apagao_out2024[['DatInicioInterrupcao','DatFimInterrupcao',
                       'IdeMotivoInterrupcao','NumUnidadeConsumidora',
                       'duracao_horas','DscFatoGeradorInterrupcao']].head(20))
print(apagao_out2024['IdeMotivoInterrupcao'].value_counts())

        DatInicioInterrupcao   DatFimInterrupcao  IdeMotivoInterrupcao  \
1673096  2024-10-11 19:22:56 2024-10-12 09:55:48                     3   
1673128  2024-10-14 09:27:06 2024-10-15 11:39:42                     6   
1673160  2024-10-14 09:27:13 2024-10-14 12:30:25                     0   
1673192  2024-10-14 09:27:22 2024-10-14 11:02:36                     0   
1673224  2024-10-14 09:27:31 2024-10-14 10:21:06                     6   
1673256  2024-10-14 09:27:47 2024-10-14 14:05:46                     0   
1673288  2024-10-14 09:28:25 2024-10-15 12:15:00                     6   
1673320  2024-10-14 09:28:28 2024-10-15 15:00:18                     6   
1673352  2024-10-14 09:28:56 2024-10-15 16:28:25                     0   
1673384  2024-10-14 09:29:00 2024-10-14 09:40:06                     6   
1673416  2024-10-14 09:29:01 2024-10-14 19:43:22                     0   
1673448  2024-10-14 09:29:09 2024-10-14 10:58:50                     6   
1673480  2024-10-14 09:29:09 2024-10-1